# JobFit AI: Optimized Kimi 2.6 Workflow

This guide uses **Kimi 2.6 through the OpenAI Python SDK** and keeps the workflow predictable:

1. Kimi creates one focused search query.
2. Python calls Olostep Search once and gets 10 results.
3. Kimi selects the best sources to scrape.
4. Python scrapes only those selected sources.
5. Kimi ranks the scraped jobs and recommends the best one to apply for.

This avoids long agent loops and repeated tool calls while still showing agentic decision-making.

In [40]:
import json
import os
import re
import requests
from IPython.display import Markdown, display
from openai import OpenAI
from pypdf import PdfReader

KIMI_MODEL = "kimi-k2.6"
KIMI_BASE_URL = "https://api.moonshot.ai/v1"
OLOSTEP_SEARCH_URL = "https://api.olostep.com/v1/searches"
OLOSTEP_SCRAPE_URL = "https://api.olostep.com/v1/scrapes"

cv_path = "abid-resume.pdf"
job_preferences = """
Remote data science, AI writer, or technical writer roles in AI, machine learning, data science, or cloud.
Prefer roles focused on technical content, tutorials, developer education, research writing, and AI product storytelling.
""".strip()

MAX_SEARCH_RESULTS = 10
MAX_SOURCES_TO_SCRAPE = 5
MAX_SCRAPED_CHARS = 9000

DIRECT_JOB_URL_HINTS = [
   "greenhouse.io",
   "boards.greenhouse.io",
   "job-boards.greenhouse.io",
   "lever.co",
   "jobs.ashbyhq.com",
   "workable.com",
   "smartrecruiters.com",
   "bamboohr.com/careers",
   "myworkdayjobs.com",
   "jobs.lever.co",
   "/careers/",
   "/jobs/",
   "/job/",
   "/positions/",
   "/openings/"
]

AGGREGATOR_URL_HINTS = [
   "indeed.com",
   "ziprecruiter.com",
   "lensa.com",
   "jobright.ai",
   "remotive.com/remote-jobs",
   "workingincontent.com",
   "dailyremote.com",
   "bebee.com",
   "talents.vaia.com",
   "builtin.com/jobs",
   "linkedin.com/jobs/search",
   "glassdoor.com",
   "monster.com"
]

kimi = OpenAI(
   api_key=os.getenv("MOONSHOT_API_KEY"),
   base_url=KIMI_BASE_URL
)

print("Setup complete.")
print("MOONSHOT_API_KEY:", "set" if os.getenv("MOONSHOT_API_KEY") else "missing")
print("OLOSTEP_API_KEY:", "set" if os.getenv("OLOSTEP_API_KEY") else "missing")
print("Kimi model:", KIMI_MODEL)
print("CV file:", cv_path)

Setup complete.
MOONSHOT_API_KEY: set
OLOSTEP_API_KEY: set
Kimi model: kimi-k2.6
CV file: abid-resume.pdf


## Helper Functions

In [41]:
def require_env(name):
   value = os.getenv(name)
   if not value:
       raise ValueError(f"Set the {name} environment variable.")
   return value


def ask_kimi(prompt, *, response_format=None):
   kwargs = {
       "model": KIMI_MODEL,
       "messages": [
           {"role": "system", "content": "You are JobFit AI, a precise job-search and job-fit assistant."},
           {"role": "user", "content": prompt}
       ],
       "extra_body": {"thinking": {"type": "disabled"}}
   }

   if response_format:
       kwargs["response_format"] = response_format

   response = kimi.chat.completions.create(**kwargs)
   return response.choices[0].message.content.strip()


def parse_json_response(text):
   cleaned = text.strip()

   if cleaned.startswith("```"):
       cleaned = re.sub(r"^```(?:json)?", "", cleaned).strip()
       cleaned = re.sub(r"```$", "", cleaned).strip()

   try:
       return json.loads(cleaned)
   except json.JSONDecodeError:
       match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
       if match:
           return json.loads(match.group(0))
       raise


def read_cv(path, max_chars=12000):
   reader = PdfReader(path)
   text = ""

   for page_number, page in enumerate(reader.pages, start=1):
       page_text = page.extract_text() or ""
       print(f"Read CV page {page_number}: {len(page_text)} characters")
       text += page_text + "\n"

   return text[:max_chars]


def olostep_headers():
   return {
       "Authorization": f"Bearer {require_env('OLOSTEP_API_KEY')}",
       "Content-Type": "application/json"
   }


def search_jobs(query, limit=10):
   safe_limit = max(1, min(int(limit), MAX_SEARCH_RESULTS))
   print(f"[search] {query!r} limit={safe_limit}")

   response = requests.post(
       OLOSTEP_SEARCH_URL,
       headers=olostep_headers(),
       json={"query": query},
       timeout=60
   )
   response.raise_for_status()

   links = response.json().get("result", {}).get("links", [])
   results = []

   for index, link in enumerate(links[:safe_limit], start=1):
       if isinstance(link, dict) and link.get("url"):
           results.append({
               "index": index,
               "title": link.get("title", "Untitled source"),
               "url": link["url"],
               "description": link.get("description", "")
           })

   return results


def looks_like_direct_job_url(url):
   normalized = url.lower()

   if any(hint in normalized for hint in AGGREGATOR_URL_HINTS):
       return False

   return any(hint in normalized for hint in DIRECT_JOB_URL_HINTS)


def filter_direct_job_sources(sources):
   direct_sources = []
   rejected_sources = []

   for source in sources:
       if looks_like_direct_job_url(source["url"]):
           direct_sources.append(source)
       else:
           rejected_sources.append({
               **source,
               "reason": "Not a direct job-listing URL or likely aggregator/search page"
           })

   return direct_sources, rejected_sources


def scrape_url(source):
   print(f"[scrape] {source['url']}")

   response = requests.post(
       OLOSTEP_SCRAPE_URL,
       headers=olostep_headers(),
       json={"url_to_scrape": source["url"], "formats": ["markdown"]},
       timeout=120
   )
   response.raise_for_status()

   markdown = response.json().get("result", {}).get("markdown_content") or ""
   return {
       **source,
       "scraped_chars": len(markdown),
       "markdown": markdown[:MAX_SCRAPED_CHARS]
   }


def table_markdown(rows, columns):
   if not rows:
       return "No rows."

   header = "| " + " | ".join(columns) + " |"
   separator = "| " + " | ".join(["---"] * len(columns)) + " |"
   body = []

   for row in rows:
       values = []
       for column in columns:
           value = str(row.get(column, "")).replace("\n", " ").replace("|", "\\|")
           values.append(value[:180])
       body.append("| " + " | ".join(values) + " |")

   return "\n".join([header, separator, *body])

print("Helpers loaded.")

Helpers loaded.


## 1. Read CV

In [42]:
cv_text = read_cv(cv_path)

print("\nCV characters used:", len(cv_text))
print("\nCV preview:\n")
print(cv_text[:1200])

Read CV page 1: 1127 characters
Read CV page 2: 1818 characters

CV characters used: 2947

CV preview:

Experience Work
Working on blog writing, editing, fact-checking, and providing
editorial support for topics like data science, deep learning, LLMs,
MLOps, AI, and career advice.
DATA SCIENCE EDITOR &
COPYWRITER
2021 - Present | DataCamp
DATA SCIENCE COPYWRITER & EDITOR
Education
Graduated with Distinction
MSc in Technology Management 
STAFFORDSHIRE UNIVERSITY, 2017
Computer Networks 
COMSATS Institute of Information Technology
BE TELECOMMUNICATION, 2014
Skills
Editing
Copywriting
Teamwork
Graphic Design
Data Analysis
ML/DL/NLP
LLMs/ MLOps
SQL
I am a Certified Professional Data
Scientist who enjoys simplifying complex
tools and concepts. Currently, I am
focusing on creating and editing content
for machine learning and data science
technologies.
About Me
Contact
Abid Ali
Awan
abid@tuta.com
1abidaliawan
abid.work
Tasks include editing and publishing blogs, quality assurance,
managing th

## 2. Kimi Creates One Search Query

In [43]:
query_prompt = f"""
Create exactly one focused web search query to find direct job listing pages for this candidate.

Candidate CV excerpt:
{cv_text[:5000]}

Job preferences:
{job_preferences}

The query must target direct job listing URLs, not aggregator search pages.
Prefer company career pages and ATS pages such as Greenhouse, Lever, Ashby, Workable, SmartRecruiters, and company /careers/ pages.
Include words like apply, careers, jobs, remote, AI, machine learning, technical writer, developer education.
Avoid terms that produce generic job boards such as Indeed, ZipRecruiter, Remotive, and WorkingInContent.

Return only the search query. No bullets, no explanation.
"""

search_query = ask_kimi(query_prompt).strip().strip('"')
print("Search query:", search_query)

Search query: site:greenhouse.io OR site:lever.co OR site:ashbyhq.com OR site:workable.com OR site:smartrecruiters.com OR site:boards.greenhouse.io OR site:jobs.lever.co "technical writer" OR "developer education" OR "AI writer" OR "machine learning" OR "data science" remote apply careers -indeed -ziprecruiter -remotive -workingincontent


## 3. Search 10 Sources Once

In [44]:
raw_search_results = search_jobs(search_query, limit=MAX_SEARCH_RESULTS)
search_results, rejected_search_results = filter_direct_job_sources(raw_search_results)

for index, source in enumerate(search_results, start=1):
   source["index"] = index

print("Raw sources:", len(raw_search_results))
print("Direct job-listing sources:", len(search_results))
print("Rejected aggregator/search sources:", len(rejected_search_results))

display(Markdown(table_markdown(search_results, ["index", "title", "url", "description"])))

if rejected_search_results:
   display(Markdown("### Rejected Sources\n\n" + table_markdown(rejected_search_results, ["title", "url", "reason"])))

[search] 'site:greenhouse.io OR site:lever.co OR site:ashbyhq.com OR site:workable.com OR site:smartrecruiters.com OR site:boards.greenhouse.io OR site:jobs.lever.co "technical writer" OR "developer education" OR "AI writer" OR "machine learning" OR "data science" remote apply careers -indeed -ziprecruiter -remotive -workingincontent' limit=10
Raw sources: 10
Direct job-listing sources: 10
Rejected aggregator/search sources: 0


| index | title | url | description |
| --- | --- | --- | --- |
| 1 | Turnitin, LLC Principal Machine Learning Scientist (US Remote) | https://jobs.smartrecruiters.com/TurnitinLLC/744000106361526-principal-machine-learning-scientist-us-remote- | You will work closely with product and engineering teams across Turnitin to integrate Machine Learning into a broad suite of learning, teaching and integrity ... |
| 2 | Job Application for Technical Writer at Sureify - Greenhouse | http://job-boards.greenhouse.io/sureify/jobs/5187353008 | In this role, you will partner closely with our existing technical writer, product managers, and engineers to build and maintain developer-facing documentation ... |
| 3 | Job Application for Technical Writer at Tailscale - Greenhouse | http://job-boards.greenhouse.io/tailscale/jobs/4684062005 | Job Description. We are looking for a Technical Writer to join our Documentation team to help us create and deliver world-class product documentation, tutorials ... |
| 4 | Turnitin, LLC Senior Machine Learning Scientist (USA Remote) | https://jobs.smartrecruiters.com/TurnitinLLC/744000111110355-senior-machine-learning-scientist-usa-remote- | Master's degree or PhD in Computer Science, Electrical Engineering, AI, Machine Learning, applied math or related field, with relevant industry experience, or ... |
| 5 | Data Scientist - AI Trainer - Freelance - 8-20hr p/w - Remote - Jobs | https://jobs.ashbyhq.com/10xteam/428bf33f-749b-4653-86cb-aeb05104bb68 | Experienced in advising on, preparing, and executing data science or machine learning projects. Skilled at evaluating analytical approaches ... |
| 6 | Job Application for Technical Content Writer at Runpod, Inc. | http://job-boards.greenhouse.io/runpod/jobs/5179915008 | As a Technical Writer at Runpod, you will treat our documentation as a core product. ... Domain Expertise: Familiarity with AI, machine learning, and GPU ... |
| 7 | Job Application for Senior Technical Writer at LaunchDarkly | http://job-boards.greenhouse.io/launchdarkly/jobs/7666736003 | Back to jobs. Senior Technical Writer. Remote - US. Apply. About the Job: LaunchDarkly is looking for someone to join our small-but-mighty docs team. This is a ... |
| 8 | Senior Machine Learning Scientist - Applied Research (USA Remote) | https://jobs.smartrecruiters.com/TurnitinLLC/744000103395782-senior-machine-learning-scientist-applied-research-usa-remote- | Machine Learning powers our AI Writing detection system, gives automated feedback on student writing, investigates authorship of student writing, revolutionizes ... |
| 9 | Job Application for Technical Writer at Turnkey - Greenhouse | http://job-boards.greenhouse.io/turnkeycareers/jobs/4226665009 | Technical Writer. United States (Remote). Apply. About Us. Turnkey is developer ... We encourage individuals of all backgrounds to apply. Compensation range. |
| 10 | Job Application for Machine Learning Engineer at TerraClear | http://job-boards.greenhouse.io/terraclear/jobs/5830566004 | As a Machine Learning Engineer, you will design, train, evaluate, and deploy deep learning models for real-world computer vision applications across mapping ... |

## 4. Kimi Selects the Best Sources to Scrape

In [45]:
selection_prompt = f"""
Select up to {MAX_SOURCES_TO_SCRAPE} direct job listing sources to scrape from these filtered search results.

Candidate preferences:
{job_preferences}

Search results:
{json.dumps(search_results, indent=2)}

Choose only real, specific job posting pages that a candidate can apply to directly.
Reject broad job boards, aggregator pages, search result pages, category pages, low-detail directories, spam, and unrelated pages.
A source must be a direct job listing or direct company/ATS job page. If there are no direct job listings, select nothing.

Return only JSON in this exact shape:
{{
  "selected": [
    {{"index": 1, "reason": "short reason"}}
  ],
  "skipped": [
    {{"index": 2, "reason": "short reason"}}
  ]
}}
"""

selection = parse_json_response(ask_kimi(selection_prompt))
selected_indexes = [item.get("index") for item in selection.get("selected", [])]

selected_sources = []
for item in selection.get("selected", []):
   selected_index = item.get("index")
   match = next((source for source in search_results if source["index"] == selected_index), None)
   if match:
       selected_sources.append({**match, "selection_reason": item.get("reason", "Selected by Kimi")})

selected_sources = selected_sources[:MAX_SOURCES_TO_SCRAPE]
skipped_sources = selection.get("skipped", [])

print("Selected sources:", len(selected_sources))
display(Markdown(table_markdown(selected_sources, ["index", "title", "url", "selection_reason"])))

Selected sources: 5


| index | title | url | selection_reason |
| --- | --- | --- | --- |
| 2 | Job Application for Technical Writer at Sureify - Greenhouse | http://job-boards.greenhouse.io/sureify/jobs/5187353008 | Direct Greenhouse ATS page for Technical Writer role with developer-facing documentation focus |
| 3 | Job Application for Technical Writer at Tailscale - Greenhouse | http://job-boards.greenhouse.io/tailscale/jobs/4684062005 | Direct Greenhouse ATS page for Technical Writer with tutorials and product documentation focus |
| 5 | Data Scientist - AI Trainer - Freelance - 8-20hr p/w - Remote - Jobs | https://jobs.ashbyhq.com/10xteam/428bf33f-749b-4653-86cb-aeb05104bb68 | Direct Ashby ATS page for Data Scientist - AI Trainer freelance remote role |
| 6 | Job Application for Technical Content Writer at Runpod, Inc. | http://job-boards.greenhouse.io/runpod/jobs/5179915008 | Direct Greenhouse ATS page for Technical Content Writer at AI/GPU cloud company |
| 7 | Job Application for Senior Technical Writer at LaunchDarkly | http://job-boards.greenhouse.io/launchdarkly/jobs/7666736003 | Direct Greenhouse ATS page for Senior Technical Writer, remote US |

## 5. Scrape Selected Sources

In [46]:
scraped_sources = []

for source in selected_sources:
   try:
       scraped_sources.append(scrape_url(source))
   except Exception as error:
       print(f"Scrape failed for {source['url']}: {error}")

print("Scraped sources:", len(scraped_sources))
display(Markdown(table_markdown(scraped_sources, ["index", "title", "url", "scraped_chars"])))

[scrape] http://job-boards.greenhouse.io/sureify/jobs/5187353008
[scrape] http://job-boards.greenhouse.io/tailscale/jobs/4684062005
[scrape] https://jobs.ashbyhq.com/10xteam/428bf33f-749b-4653-86cb-aeb05104bb68
[scrape] http://job-boards.greenhouse.io/runpod/jobs/5179915008
[scrape] http://job-boards.greenhouse.io/launchdarkly/jobs/7666736003
Scraped sources: 5


| index | title | url | scraped_chars |
| --- | --- | --- | --- |
| 2 | Job Application for Technical Writer at Sureify - Greenhouse | http://job-boards.greenhouse.io/sureify/jobs/5187353008 | 6508 |
| 3 | Job Application for Technical Writer at Tailscale - Greenhouse | http://job-boards.greenhouse.io/tailscale/jobs/4684062005 | 15246 |
| 5 | Data Scientist - AI Trainer - Freelance - 8-20hr p/w - Remote - Jobs | https://jobs.ashbyhq.com/10xteam/428bf33f-749b-4653-86cb-aeb05104bb68 | 3858 |
| 6 | Job Application for Technical Content Writer at Runpod, Inc. | http://job-boards.greenhouse.io/runpod/jobs/5179915008 | 12196 |
| 7 | Job Application for Senior Technical Writer at LaunchDarkly | http://job-boards.greenhouse.io/launchdarkly/jobs/7666736003 | 15705 |

## 6. Kimi Ranks Jobs and Picks the Best One

In [47]:
analysis_input = [
   {
       "title": item["title"],
       "url": item["url"],
       "selection_reason": item.get("selection_reason", ""),
       "scraped_chars": item["scraped_chars"],
       "markdown": item["markdown"]
   }
   for item in scraped_sources
]

analysis_prompt = f"""
Rank these scraped direct job listings for the candidate.

Candidate CV:
{cv_text}

Job preferences:
{job_preferences}

Scraped sources:
{json.dumps(analysis_input, ensure_ascii=False, indent=2)}

Write a clean, readable Markdown report.

Formatting rules:
- Use valid Markdown headings with #, ##, and ###.
- Put a blank line after every heading.
- Use a Markdown table for the ranked summary.
- Use bullet lists for reasons, concerns, and application angles.
- Keep each bullet short and specific.
- Do not write long paragraph blocks inside the ranking details.
- Do not merge labels on one line. Each field must be on its own line.
- Include clickable Markdown links for URLs: [Apply or view job](URL).
- Do not recommend aggregator pages, search pages, or broad job-board category pages as jobs to apply for.
- Only rank direct job listing URLs. If a scraped source is not a direct job listing, move it to Skipped Or Weak Sources.

Return exactly this structure:

# JobFit AI Results

## Best Job To Apply For

**Role:** <job title>  
**Company:** <company>  
**Fit Score:** <score>/100  
**URL:** [Apply or view job](<url>)

**Why this is the best fit:**

- <specific reason 1>
- <specific reason 2>
- <specific reason 3>

## Ranked Summary

| Rank | Role / Source | Company | Fit | Recommendation |
| --- | --- | --- | --- | --- |
| 1 | <role> | <company> | <score>/100 | Apply / Maybe / Skip |

## Detailed Notes

### 1. <Job title or source title>

**Company:** <company>  
**Fit Score:** <score>/100  
**Recommendation:** Apply / Maybe / Skip  
**URL:** [Apply or view job](<url>)

**Why it matches:**

- <bullet>
- <bullet>

**Concerns:**

- <bullet>
- <bullet>

**Application angle:**

- <bullet>
- <bullet>

## Skipped Or Weak Sources

- **<source title>:** <why it is weak, generic, inaccessible, or not worth applying directly>
"""

final_report = ask_kimi(analysis_prompt)
display(Markdown(final_report))

# JobFit AI Results

## Best Job To Apply For

**Role:** Technical Content Writer  
**Company:** Runpod, Inc.  
**Fit Score:** 94/100  
**URL:** [Apply or view job](http://job-boards.greenhouse.io/runpod/jobs/5179915008)

**Why this is the best fit:**

- Direct intersection of AI/ML infrastructure and technical content creation, matching candidate's core expertise in MLOps, LLMs, and GPU/cloud technologies
- Explicitly values docs-as-code, Markdown, and Git workflows that align with candidate's existing technical stack
- Strong emphasis on tutorials, quickstarts, and developer education—exactly matching stated preference for "technical content, tutorials, developer education"
- AI-augmented workflow requirement matches candidate's hands-on experience with LLMs and AI tools (ChatGPT book, fine-tuning projects)
- Remote-first with competitive compensation ($120K-$180K) and meaningful equity

## Ranked Summary

| Rank | Role / Source | Company | Fit | Recommendation |
| --- | --- | --- | --- | --- |
| 1 | Technical Content Writer | Runpod, Inc. | 94/100 | Apply |
| 2 | Technical Writer | Tailscale | 87/100 | Apply |
| 3 | Technical Writer | Sureify | 72/100 | Maybe |
| 4 | Senior Technical Writer | LaunchDarkly | 68/100 | Maybe |
| 5 | Data Scientist - AI Trainer | 10x Team | 55/100 | Skip |

## Detailed Notes

### 1. Technical Content Writer

**Company:** Runpod, Inc.  
**Fit Score:** 94/100  
**Recommendation:** Apply  
**URL:** [Apply or view job](http://job-boards.greenhouse.io/runpod/jobs/5179915008)

**Why it matches:**

- Domain expertise in AI/ML infrastructure, GPU cloud, and MLOps directly matches candidate's portfolio (MLOps CI/CD project, Stable Diffusion fine-tuning, LLaMA 2 fine-tuning)
- Requires creating tutorials, quickstarts, and API documentation—candidate's proven strength at DataCamp and KDnuggets
- Explicit AI literacy requirement; candidate wrote book on ChatGPT productivity and has multiple LLM projects
- Docs-as-code workflow (Markdown/Git) matches modern publishing tools candidate likely uses
- Developer education focus aligns with "AI product storytelling" preference

**Concerns:**

- Requires 3+ years in cloud/AI infrastructure specifically; candidate's experience is more editorial/content-heavy
- Bachelor's in CS/Engineering preferred; candidate has Technology Management MSc and Telecommunications BE
- May require deeper systems-level distributed systems knowledge than demonstrated

**Application angle:**

- Lead with MLOps project portfolio and GPU-based ML work (Stable Diffusion, LLaMA 2 fine-tuning on cloud)
- Emphasize DataCamp/KDnuggets experience creating technical tutorials for developers
- Highlight book on ChatGPT and AI-augmented writing workflow as direct "AI literacy" evidence

---

### 2. Technical Writer

**Company:** Tailscale  
**Fit Score:** 87/100  
**Recommendation:** Apply  
**URL:** [Apply or view job](http://job-boards.greenhouse.io/tailscale/jobs/4684062005)

**Why it matches:**

- Strong tutorial and conceptual guide focus matches candidate's preference for "technical content, tutorials, developer education"
- Explicitly values networking/security concepts; candidate has CCNA certification and Telecommunications background
- Docs-as-code, Git, Markdown requirements align with modern editorial workflows
- Interest in LLMs/automation for content matches candidate's AI tool expertise
- Remote-first with inclusive culture and professional development budget

**Concerns:**

- Networking/security focus is adjacent to, not core of, candidate's recent AI/ML specialization
- Requires "hands-on experience with networking or security products"; candidate's recent work is more content/editorial
- Software development experience is "nice to have" but candidate has limited coding depth beyond ML projects

**Application angle:**

- Highlight CCNA certification and telecommunications engineering foundation as networking credibility
- Emphasize ability to write for diverse technical audiences (developers, infrastructure engineers, security practitioners)
- Showcase community engagement experience (KDnuggets event organization, open-source contributions)

---

### 3. Technical Writer

**Company:** Sureify  
**Fit Score:** 72/100  
**Recommendation:** Maybe  
**URL:** [Apply or view job](http://job-boards.greenhouse.io/sureify/jobs/5187353008)

**Why it matches:**

- Developer-facing documentation and API specifications align with technical writing preference
- Information architecture and knowledge management interest matches editorial organization skills
- Cross-functional collaboration requirement fits candidate's team management experience at KDnuggets

**Concerns:**

- Life insurance/annuity industry is far from candidate's AI/ML domain expertise
- Enterprise B2B SaaS focus without AI/ML component wastes candidate's specialized knowledge
- TypeScript/React and SI partner documentation are significant gaps in candidate background
- Scrum Master "nice to have" suggests potential scope creep away from pure writing

**Application angle:**

- Frame editorial team management at KDnuggets as cross-functional collaboration evidence
- Downplay industry mismatch by emphasizing transferable documentation architecture skills
- Consider only if candidate wants to diversify away from AI/ML specialization

---

### 4. Senior Technical Writer

**Company:** LaunchDarkly  
**Fit Score:** 68/100  
**Recommendation:** Maybe  
**URL:** [Apply or view job](http://job-boards.greenhouse.io/launchdarkly/jobs/7666736003)

**Why it matches:**

- AI Configs product area offers some AI relevance
- Strong technical writing and editing skills requirement matches core competency
- SDK/API documentation experience transferable from current role

**Concerns:**

- "Senior" level with 5+ years requirement may exceed candidate's formal technical writing tenure
- Strong technical background "possibly as software developer" is a stretch; candidate is writer-first
- DevOps/feature flag domain is not candidate's demonstrated expertise area
- Higher compensation bands suggest more senior candidates will compete

**Application angle:**

- Emphasize depth of technical editing at DataCamp/KDnuggets over years of experience
- Highlight AI Configs relevance with LLM/MLOps project portfolio
- Consider if willing to stretch into more engineering-heavy documentation role

---

### 5. Data Scientist - AI Trainer

**Company:** 10x Team  
**Fit Score:** 55/100  
**Recommendation:** Skip  
**URL:** [Apply or view job](https://jobs.ashbyhq.com/10xteam/428bf33f-749b-4653-86cb-aeb05104bb68)

**Why it matches:**

- Data science domain expertise requirement matches candidate's credentials
- Freelance flexibility (8-20 hrs/week) could supplement existing roles

**Concerns:**

- Role is evaluating AI outputs, not creating content—misaligned with candidate's stated preference for "technical content, tutorials, developer education, research writing, and AI product storytelling"
- Requires "senior-level data scientist with significant professional experience"; candidate's professional data science experience is primarily editorial/content
- EU/UK location requirement may conflict with candidate's unstated location (Pakistani institutions in CV)
- Freelance contract structure doesn't match preference for stable remote role
- Hourly rate (€88-€121) is attractive but role fundamentally misaligned with career trajectory

**Application angle:**

- Not recommended; candidate's strengths are in content creation and editorial leadership, not AI evaluation/QA

## Skipped Or Weak Sources

- None. All five scraped sources are direct job listings on company ATS platforms (Greenhouse, Ashby) with apply buttons and specific role requirements.

## Optional: Inspect Variables

In [48]:
print("search_query:", search_query)
print("search_results:", len(search_results))
print("selected_sources:", len(selected_sources))
print("scraped_sources:", len(scraped_sources))

search_query: site:greenhouse.io OR site:lever.co OR site:ashbyhq.com OR site:workable.com OR site:smartrecruiters.com OR site:boards.greenhouse.io OR site:jobs.lever.co "technical writer" OR "developer education" OR "AI writer" OR "machine learning" OR "data science" remote apply careers -indeed -ziprecruiter -remotive -workingincontent
search_results: 10
selected_sources: 5
scraped_sources: 5
